<a href="https://colab.research.google.com/github/lawrence-kagugo/kenya-telecom-churn-analysis/blob/main/notebooks/03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kenya Telecom Customer Churn - Data Quality Assessment
**Step:** 6 - Data Quality Assessment

In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/lawrence-kagugo/kenya-telecom-churn-analysis/refs/heads/main/data/raw/kenya_telecom_customer_churn_dataset.csv"
df = pd.read_csv(url)
print("Loaded:", df.shape)

Loaded: (5000, 70)


In [8]:
# duplicated() flags rows that are exact copies of another row.
# .sum() counts how many True (duplicate) flags there are.
# We check both: duplicate CustomerID (should be unique, it's our primary key)
# and fully duplicate rows (a stronger, rarer signal of a data pipeline error).
print("Duplicate CustomerIDs:", df['CustomerID'].duplicated().sum())
print("Fully duplicate rows:", df.duplicated().sum())

Duplicate CustomerIDs: 0
Fully duplicate rows: 0


**Finding 1: No duplicates**

Zero duplicate CustomerIDs and zero fully duplicate rows found. CustomerID is
confirmed as a reliable unique identifier for this dataset.

In [9]:
# Check 1: FeaturesUsed should never exceed FeaturesAvailable — you can't use
# more features than the plan actually offers. This is a logical impossibility
# if it occurs, not just an outlier.
invalid_features = (df['FeaturesUsed'] > df['FeaturesAvailable']).sum()
print("Rows where FeaturesUsed > FeaturesAvailable:", invalid_features)

# Check 2: For churned customers, ChurnDate should always be AFTER JoinDate —
# you can't churn before you became a customer. We need to convert to datetime
# first since we know from Step 5 these are stored as text (object) currently.
df['JoinDate_dt'] = pd.to_datetime(df['JoinDate'])
df['ChurnDate_dt'] = pd.to_datetime(df['ChurnDate'])

churned = df[df['Churn'] == 'Yes']
invalid_churn_dates = (churned['ChurnDate_dt'] < churned['JoinDate_dt']).sum()
print("Churned customers with ChurnDate before JoinDate:", invalid_churn_dates)

Rows where FeaturesUsed > FeaturesAvailable: 0
Churned customers with ChurnDate before JoinDate: 0


**Finding 2: Logical consistency checks pass**

Zero rows where FeaturesUsed exceeds FeaturesAvailable. Zero churned customers
with a ChurnDate before their JoinDate. Both checks confirm internal logical
consistency in the dataset.

In [10]:
# Score columns should logically fall between 0 and 100.
score_cols = ['ChurnRiskScore', 'CustomerHealthScore', 'EngagementScore', 'FeatureAdoptionRate']
for col in score_cols:
    out_of_range = ((df[col] < 0) | (df[col] > 100)).sum()
    print(f"{col}: min={df[col].min():.1f}, max={df[col].max():.1f}, out-of-range count={out_of_range}")

print()

# Financial columns should never be negative.
money_cols = ['MonthlyCharges', 'TotalCharges', 'CustomerLifetimeValue', 'OutstandingBalance', 'EstimatedAnnualRevenue']
for col in money_cols:
    negatives = (df[col] < 0).sum()
    print(f"{col}: min={df[col].min():.1f}, negative count={negatives}")

ChurnRiskScore: min=0.0, max=100.0, out-of-range count=0
CustomerHealthScore: min=28.9, max=100.0, out-of-range count=0
EngagementScore: min=0.0, max=100.0, out-of-range count=0
FeatureAdoptionRate: min=0.0, max=100.0, out-of-range count=0

MonthlyCharges: min=500.0, negative count=0
TotalCharges: min=0.0, negative count=0
CustomerLifetimeValue: min=5000.0, negative count=0
OutstandingBalance: min=0.0, negative count=0
EstimatedAnnualRevenue: min=6000.0, negative count=0


**Finding 3: Valid ranges confirmed**

All score columns (ChurnRiskScore, CustomerHealthScore, EngagementScore,
FeatureAdoptionRate) fall within the expected 0-100 range. All financial columns
(MonthlyCharges, TotalCharges, CustomerLifetimeValue, OutstandingBalance,
EstimatedAnnualRevenue) contain no negative values. Actual min values (e.g.
MonthlyCharges min = 500) suggest realistic business floors rather than data errors.

In [11]:
# IQR (Interquartile Range) method: identifies values that fall far outside
# the "normal" spread of the data. Q1 = 25th percentile, Q3 = 75th percentile.
# Anything below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is flagged as a statistical outlier.
cols_to_check = ['MonthlyCharges', 'TotalCharges', 'CustomerLifetimeValue', 'OutstandingBalance']

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers ({outliers/len(df)*100:.1f}%), normal range=({lower:.0f} to {upper:.0f})")

MonthlyCharges: 216 outliers (4.3%), normal range=(-4380 to 10935)
TotalCharges: 387 outliers (7.7%), normal range=(-208761 to 441768)
CustomerLifetimeValue: 324 outliers (6.5%), normal range=(-145019 to 339768)
OutstandingBalance: 440 outliers (8.8%), normal range=(-782 to 1393)


In [12]:
# Check whether "outlier" CustomerLifetimeValue rows are concentrated in
# specific segments/plans, which would explain them as legitimate business
# variation rather than data errors.
Q1 = df['CustomerLifetimeValue'].quantile(0.25)
Q3 = df['CustomerLifetimeValue'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

outlier_rows = df[df['CustomerLifetimeValue'] > upper]
print("SubscriptionPlan breakdown of CLV outliers:")
print(outlier_rows['SubscriptionPlan'].value_counts())
print()
print("CustomerSegment breakdown of CLV outliers:")
print(outlier_rows['CustomerSegment'].value_counts())

SubscriptionPlan breakdown of CLV outliers:
SubscriptionPlan
Business      152
Enterprise    148
Unlimited      14
Premium         7
Family          3
Name: count, dtype: int64

CustomerSegment breakdown of CLV outliers:
CustomerSegment
Corporate      164
SME             85
Government      67
Residential      8
Name: count, dtype: int64


**Finding 4: Outliers reflect legitimate business variation, not errors**

IQR method flagged 4-9% of rows as outliers in MonthlyCharges, TotalCharges,
CustomerLifetimeValue, and OutstandingBalance. Investigation showed CLV outliers
are concentrated in Business/Enterprise plans (92% of outliers) and Corporate
segment (over half of outliers) - consistent with expected higher value for
premium/corporate customers, not data entry errors. Decision: retain these
records as-is; do not cap or remove. Standard IQR outlier detection is not
appropriate for this mixed-population data without segmenting first.